# Выбор метрики для Bank Marketing

## Структура
1. Бизнес-контекст
2. Дисбаланс классов
3. вопросы для выбора
4. Бизнес-вопрос - метрика, как связаны
5. Роль baseline-модели
6. Выбор метрики
7. Почему не другие метрики
8. Ограничения выбранной метрики
9. Что скажем бизнесу
10. Итоговый вывод

## 1. Бизнес-контекст

## описание кейса
Необходимо предсказать, согласится ли клиент открыть срочный вклад по итогам телемаркетинговой кампании, используя демографические признаки, историю контактов и экономические индикаторы

- задача - **бинарная классификация**: `yes` (согласится) / `no` (не согласится)
- в условии **нет** ограничения бюджета, стоимости звонка, прибыли депозита или лимита K
- значит, метрика должна оценивать **качество предсказания**

## 2. Дисбаланс классов

в данных 88.73% ответов `no` и 11.27% ответов `yes`

это сильный дисбаланс. Класс `yes` (клиенты, готовые оформить депозит) - редкий, но именно он важен для бизнеса

### что это значит для выбора метрики
- модель, которая всегда говорит `no`, даёт Accuracy около 88.7%. Это высокий показатель, но модель бесполезна: она не находит ни одного клиента
- метрика должна делать дисбаланс заметным. Accuracy и ROC-AUC этого не делают
- baseline для случайного предсказания на положительном классе = prevalence = 0.1127. Любая метрика, которая это не учитывает, будет обманывать

## 3. Вопросы для выбора

### 1. Какое бизнес-решение принимает модель?
модель предсказывает вероятность подписки. Банк на основе этих вероятностей решает, кому звонить. Это не автоматическое решение, а приоритизация

### 2. Каков доступный бюджет кампании?
в условии кейса бюджет **не ограничен**. Банк может обзвонить любое число клиентов

### 3. Сколько стоит один звонок?
в условии кейса стоимость звонка **не задана**. мы не делаем допущений о деньгах

### 4. Сколько приносит один оформленный депозит?
в условии кейса прибыль от депозита **не задана**. мы не делаем допущений о деньгах

### 5. Что дороже для банка: пропустить клиента или сделать лишний звонок?
без данных о стоимости и прибыли нельзя однозначно ответить. Но бизнес-цель - **найти как можно больше клиентов**, поэтому **Recall важен**. При этом Precision тоже важен, чтобы не звонить всем подряд

### 6. Какая допустимая доля ложных срабатываний?
без ограничения бюджета **жёсткого лимита нет**, но слишком низкий Precision означает, что модель звонит многим клиентам, которые откажутся

### 7. Какой минимально необходимый уровень охвата реальных клиентов?
бизнес хочет **высокий Recall** - найти как можно больше реальных подписчиков

### 8. Нужны вероятности или ранжирование?
достаточно **ранжирования** (модель выдаёт score, банк сортирует). Но калибровка вероятностей полезна для интерпретации

### 9. Как измерять эффект на продакшене?
сравнивать с **контрольной группой** (random selection) и **предыдущей кампанией**. Основная метрика - PR-AUC (сравнение моделей) и Precision/Recall на оптимальном пороге

### 10. Нужна ли калибровка вероятностей?
для **ранжирования** - нет, а для **интерпретации** - да. Логистическая регрессия калибрована сразу, LightGBM - нужно проверять

### 11. Подходит ли метрика к временной структуре данных?
данные собраны за несколько лет. Идеально time-based split? или StratifiedKFold

### 12. Не использует ли метрика информацию, недоступную в момент прогноза?
`duration` удалён (leak). `pdays` заменён на `was_contacted`. PR-AUC использует только предсказания

### 13. Можно ли интерпретировать метрику в бизнес-терминах?
PR-AUC - плохо (площадь под кривой). Precision и Recall - хорошо: из тех, кому позвонили, сколько оформили и из всех реальных клиентов, сколько нашли

### 14. Какая метрика позволит сравнивать модели на одном наборе?
**PR-AUC** - да, это интегральная метрика, не зависящая от порога

### 15. Какой порог зафиксирован?
**Порог не фиксирован**. Он выбирается по компромиссу Precision/Recall или по бизнес-требованиям

## 4. Связь "бизнес-вопрос - метрика"

| бизнес-вопрос | что важно | метрика |
|---|---|---|
| насколько модель лучше случайной? | сравнение моделей | PR-AUC |
| из тех, кому позвонили, сколько оформили? | точность | Precision |
| из всех реальных клиентов, сколько нашли? | полнота | Recall |

каждая метрика отвечает на свой бизнес-вопрос. Не выбираем одну метрику, а используем набор, где каждая закрывает свой уровень решения

## 5. Роль baseline-модели

Baseline-модель - Logistic Regression с `class_weight='balanced'`

### что дает
- калиброванные вероятности (а не только score). Значит, Precision и Recall на пороге считаются корректно
- интерпретируемость: видно, какие признаки важны (например, `was_contacted`), и это согласуется с бизнес-логикой
- отправную точку для сравнения: если LightGBM не лучше LR по PR-AUC, то усложнять модель нет смысла

### ограничения
LR - линейная модель, она не справляется со сложными взаимодействиями, поэтому метрика должна быть устойчивой к среднему качеству модели. PR-AUC это позволяет - он не требует идеального разделения классов, а измеряет качество на всём диапазоне порогов

### что это значит для выбора метрики
- PR-AUC подходит, потому что не требует идеального качества модели
- Precision и Recall подходят, потому что LR даёт калиброванные вероятности
- Accuracy и ROC-AUC не подходят, потому что обманывают при дисбалансе

## 6. Выбор метрики

| метрика | что измеряет | бизнес-смысл | роль в кейсе |
|---|---|---|---|
| **PR-AUC** | качество на положительном классе | - | сравнение моделей |
| **Precision** | точность на пороге | сколько оформили из тех, кому позвонили | оценка качества |
| **Recall** | полнота на пороге | сколько реальных клиентов нашли | бизнес-цель |

### вывод
- PR-AUC - для сравнения моделей (честная на дисбалансе, baseline = prevalence)
- Precision и Recall - для оценки качества на выбранном пороге
- порог выбирается по компромиссу Precision/Recall или по бизнес-требованиям

## 7. Почему не другие метрики

### Accuracy
Accuracy не различает полезную модель и бесполезную константу

### ROC-AUC
при 36000 TN даже 100 FP дают FPR ≈ 0.003. Разница между хорошей и слабой моделью сглаживается

### F1
симметрична: одинаково штрафует FP и FN. В бизнесе FN и FP имеют разную цену (найти клиента важнее, чем не позвонить лишний раз), поэтому F1 не отражает экономику задачи

## 8. Ограничения выбранной метрики

- **PR-AUC зависит от prevalence.** нельзя сравнивать PR-AUC между разными датасетами, только внутри одного
- **Precision и Recall зависят от порога.** при разных порогах - разные значения. Порог нужно фиксировать до сравнения моделей
- **PR-AUC не интерпретируется бизнесом напрямую.** он нужен для сравнения моделей, а не для отчёта бизнесу

## 12. Что скажем бизнесу

юизнесу не нужен PR-AUC. Бизнесу нужно:

- модель находит X% всех реальных клиентов (Recall).
- из тех, кому позвонили, Y% оформили депозит (Precision).
- модель в Z раз лучше случайного предсказания (сравнение PR-AUC с prevalence).

PR-AUC - для дс (сравнить модели). Precision и Recall - для бизнеса (принять решение).

это ключевое различие: техническая метрика и бизнес-метрика - разные вещи. Используем обе, но для разных целей.

## 13. Итоговый вывод

### метрика выбрана по бизнес-задаче
- банк предсказывает вероятность подписки и решает, кому звонить
- в условии кейса **нет ограничения бюджета**
- в условии кейса **нет данных о стоимости звонка и прибыли депозита**, поэтому мы не считаем Expected Profit
- Baseline-модель - LR, она даёт калиброванные вероятности

### какие метрики используем
| метрика | роль |
|---|---|
| **PR-AUC** | сравнение моделей (честная на дисбалансе, baseline = prevalence) |
| **Precision** | оценка качества на пороге (сколько оформили из тех, кому позвонили) |
| **Recall** | бизнес-цель (сколько реальных клиентов нашли) |

### что не используем
- **Accuracy** - обманывает при дисбалансе
- **ROC-AUC** - сглаживает разницу
- **F1** - симметрична, не учитывает бизнес-стоимость ошибок